# Forward models for simulations

To avoid the “inverse crime,” the forward solution used to generate the EEG simulations and the forward solution used to solve the inverse problem should include modeling errors, as is the case in reality. To this end, we generate two forward solutions with different conductivities. Here, we compare the two resulting solutions.

## 🧪 - Experiment setup
- `src`: Source spaces
- `ave`: Average recording
- `bem`: BEM layers

In [1]:
import mne
import mindmap as mm

# data manager
dm = mm.io.TutoManager("../../config.yml")
sub = "simf10"

# data
ave = dm.load(sub, "ave-temp")
trans = dm.load(sub, "eeg-trans")
src = dm.load(sub, "src")

## 🧬 - Data Processing
Forward solution for simulations.

In [2]:
# BEM model
model = mne.make_bem_model(
    subject = f"sub-{sub}",
    subjects_dir = dm.roots["fs"],
    conductivity = [0.3, 0.006, 0.3],    # conductivities for simulation (brain, skull, skin)
    ico = None,
)

bem  = mne.make_bem_solution(model)

# Forward solution
forward_sim = mne.make_forward_solution(
    info = ave.info,
    trans = trans,
    src = src,
    bem = bem,
    meg = False,
    eeg = True,
    verbose = "warning",
)

Creating the BEM geometry...
outer skin  CM is  -1.27   5.47  -8.03 mm
outer skull CM is  -0.83   4.76   8.69 mm
inner skull CM is  -0.11  -6.49  24.90 mm
Checking that surface outer skull is inside surface outer skin  ...
Checking that surface inner skull is inside surface outer skull ...
Checking distance between outer skin  and outer skull surfaces...
Minimum distance between the outer skin  and outer skull surfaces is approximately    3.9 mm
Checking distance between outer skull and inner skull surfaces...
Minimum distance between the outer skull and inner skull surfaces is approximately    0.3 mm
Surfaces passed the basic topology checks.
Complete.

Three-layer model surfaces loaded.
Computing the linear collocation solution...
    Matrix coefficients...
        outer skin  (10242) -> outer skin  (10242) ...


/tmp/ipykernel_7521/1927226005.py:2: RuntimeWarning: The bem surfaces have 10242 data points. 5120 (ico grade=4) should be enough. Dense 3-layer bems may not save properly.
  model = mne.make_bem_model(
/tmp/ipykernel_7521/1927226005.py:9: RuntimeWarning: The bem surfaces have 10242 data points. 5120 (ico grade=4) should be enough. Dense 3-layer bems may not save properly.
  bem  = mne.make_bem_solution(model)


        outer skin  (10242) -> outer skull (10242) ...
        outer skin  (10242) -> inner skull (10242) ...
        outer skull (10242) -> outer skin  (10242) ...
        outer skull (10242) -> outer skull (10242) ...
        outer skull (10242) -> inner skull (10242) ...
        inner skull (10242) -> outer skin  (10242) ...
        inner skull (10242) -> outer skull (10242) ...
        inner skull (10242) -> inner skull (10242) ...
    Inverting the coefficient matrix...
IP approach required...
    Matrix coefficients (homog)...
        inner skull (10242) -> inner skull (10242) ...
    Inverting the coefficient matrix (homog)...
    Modify the original solution to incorporate IP approach...
        Combining...
        Scaling...
Solution ready.
BEM geometry computations complete.


Forward solution for inverse problems.

In [3]:
# BEM model for inverse modeling
model = mne.make_bem_model(
    subject = f"sub-{sub}",
    subjects_dir = dm.roots["fs"],
    conductivity = [0.3, 0.012, 0.3],    # conductivities for inverse modeling (brain, skull, skin)
    ico = None,
)

bem  = mne.make_bem_solution(model)

# Forward solution
forward_inv = mne.make_forward_solution(
    info = ave.info,
    trans = trans,
    src = src,
    bem = bem,
    meg = False,
    eeg = True,
    verbose = "warning",
)

Creating the BEM geometry...
outer skin  CM is  -1.27   5.47  -8.03 mm
outer skull CM is  -0.83   4.76   8.69 mm
inner skull CM is  -0.11  -6.49  24.90 mm
Checking that surface outer skull is inside surface outer skin  ...
Checking that surface inner skull is inside surface outer skull ...
Checking distance between outer skin  and outer skull surfaces...
Minimum distance between the outer skin  and outer skull surfaces is approximately    3.9 mm
Checking distance between outer skull and inner skull surfaces...
Minimum distance between the outer skull and inner skull surfaces is approximately    0.3 mm
Surfaces passed the basic topology checks.
Complete.

Three-layer model surfaces loaded.
Computing the linear collocation solution...
    Matrix coefficients...
        outer skin  (10242) -> outer skin  (10242) ...


/tmp/ipykernel_7521/3281726356.py:2: RuntimeWarning: The bem surfaces have 10242 data points. 5120 (ico grade=4) should be enough. Dense 3-layer bems may not save properly.
  model = mne.make_bem_model(
/tmp/ipykernel_7521/3281726356.py:9: RuntimeWarning: The bem surfaces have 10242 data points. 5120 (ico grade=4) should be enough. Dense 3-layer bems may not save properly.
  bem  = mne.make_bem_solution(model)


        outer skin  (10242) -> outer skull (10242) ...
        outer skin  (10242) -> inner skull (10242) ...
        outer skull (10242) -> outer skin  (10242) ...
        outer skull (10242) -> outer skull (10242) ...
        outer skull (10242) -> inner skull (10242) ...
        inner skull (10242) -> outer skin  (10242) ...
        inner skull (10242) -> outer skull (10242) ...
        inner skull (10242) -> inner skull (10242) ...
    Inverting the coefficient matrix...
IP approach required...
    Matrix coefficients (homog)...
        inner skull (10242) -> inner skull (10242) ...
    Inverting the coefficient matrix (homog)...
    Modify the original solution to incorporate IP approach...
        Combining...
        Scaling...
Solution ready.
BEM geometry computations complete.


## 📊 - Visualization

Topography differences of a given dipole.

In [ ]:
mag = mm.anatomy.forward.compute_fwd_mag(forward_sim, forward_inv)
rdm = mm.anatomy.forward.compute_fwd_rdm(forward_sim, forward_inv)
print(f"Magnitude Magnification (MAG): {mag}")
print(f"Relative Difference Magnitude (RDM): {rdm}")

mm.viz.plot_fwd_topo_comp(forward_sim, forward_inv, 500, ave.info)

Lead vector correlation per dipole.

In [ ]:
mm.viz.plot_fwd_src_comp(forward_sim, forward_inv, forward_sim["src"])